In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
import json
import os

# Repository root (chdir so relative paths work from notebooks/)
_walk = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_walk, 'experiments')) and os.path.isdir(os.path.join(_walk, 'data')):
        PROJECT_ROOT = _walk
        break
    _walk = os.path.dirname(_walk)
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
os.chdir(PROJECT_ROOT)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import warnings
import time
from datetime import datetime
from urllib.parse import unquote

# Suppress tokenizers parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================
# EXPERIMENT CONFIGURATIONS (8 combinations)
# ============================================
EXPERIMENTS = [
    {"name": "ES5_Drop0.4_WD1e-5", "patience": 5, "dropout": 0.4, "weight_decay": 1e-5},
    {"name": "ES5_Drop0.4_WD1e-4", "patience": 5, "dropout": 0.4, "weight_decay": 1e-4},
    {"name": "ES5_Drop0.5_WD1e-5", "patience": 5, "dropout": 0.5, "weight_decay": 1e-5},
    {"name": "ES5_Drop0.5_WD1e-4", "patience": 5, "dropout": 0.5, "weight_decay": 1e-4},
    {"name": "ES7_Drop0.4_WD1e-5", "patience": 7, "dropout": 0.4, "weight_decay": 1e-5},
    {"name": "ES7_Drop0.4_WD1e-4", "patience": 7, "dropout": 0.4, "weight_decay": 1e-4},
    {"name": "ES7_Drop0.5_WD1e-5", "patience": 7, "dropout": 0.5, "weight_decay": 1e-5},
    {"name": "ES7_Drop0.5_WD1e-4", "patience": 7, "dropout": 0.5, "weight_decay": 1e-4},
]

# Fixed hyperparameters
SUBSET_RATIO = 0.5  # Use 50% of data for reliability
NUM_EPOCHS = 20  # Enough to see overfitting
LEARNING_RATE = 1e-4  # Fixed for now (will tune later)
BATCH_SIZE = 16  # Fixed for now (will tune later)
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Output directories
EXPERIMENT_ROOT = "experiments/overfitting_50pct"
METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics")
ARTIFACTS_DIR = os.path.join(EXPERIMENT_ROOT, "artifacts")
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "experiments"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "models"), exist_ok=True)

print(f"✅ Configuration loaded:")
print(f"   Total experiments: {len(EXPERIMENTS)}")
print(f"   Subset ratio: {SUBSET_RATIO*100}%")
print(f"   Max epochs: {NUM_EPOCHS}")
print(f"   Experiment: {EXPERIMENT_ROOT}")


/home/ding-zhang/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
✅ Configuration loaded:
   Total experiments: 8
   Subset ratio: 50.0%
   Max epochs: 20
   Results directory: overfitting_test_results_50pct


## 2. Load Data and Create Subset


In [2]:
# Load caption dataset
print("Loading caption dataset...")
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'caption_dataset_final_full.csv'))

# Filter for successful captions
df_success = df[df['status'] == 'success'].copy()
print(f"Total images with captions: {len(df_success)}")

# Extract style categories
df_success['style'] = df_success['style'].str.strip()  # Clean whitespace
all_styles = sorted(df_success['style'].unique())
style_to_idx = {style: idx for idx, style in enumerate(all_styles)}
idx_to_style = {idx: style for style, idx in style_to_idx.items()}
num_classes = len(all_styles)

print(f"\nStyle categories: {num_classes}")
print(f"Styles: {all_styles}")

# Create stratified subset (50%)
if SUBSET_RATIO < 1.0:
    print(f"\nCreating {SUBSET_RATIO*100}% stratified subset...")
    # Check if we have enough samples per class for stratification
    min_samples_per_class = df_success['style'].value_counts().min()
    required_samples = int(np.ceil(1 / SUBSET_RATIO))  # At least this many per class needed
    
    if min_samples_per_class < required_samples:
        print(f"⚠ Warning: Some classes have fewer than {required_samples} samples.")
        print("Using random sampling instead of stratified...")
        df_subset = df_success.sample(frac=SUBSET_RATIO, random_state=RANDOM_SEED).reset_index(drop=True)
    else:
        df_subset, _ = train_test_split(
            df_success,
            train_size=SUBSET_RATIO,
            stratify=df_success['style'],
            random_state=RANDOM_SEED
        )
    print(f"Subset size: {len(df_subset)} ({len(df_subset)/len(df_success)*100:.1f}%)")
else:
    print("\nUsing full dataset (100%)")
    df_subset = df_success.copy()
    print(f"Full dataset size: {len(df_subset)}")

# Create stratified train/val/test splits from subset
print(f"\nCreating train/val/test splits...")
print(f"  Train: {TRAIN_RATIO*100}%, Val: {VAL_RATIO*100}%, Test: {TEST_RATIO*100}%")

train_df, temp_df = train_test_split(
    df_subset,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df_subset['style'],
    random_state=RANDOM_SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_df['style'],
    random_state=RANDOM_SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(train_df)} ({len(train_df)/len(df_subset)*100:.1f}%)")
print(f"  Val: {len(val_df)} ({len(val_df)/len(df_subset)*100:.1f}%)")
print(f"  Test: {len(test_df)} ({len(test_df)/len(df_subset)*100:.1f}%)")

# Verify stratification
print(f"\nStyle distribution check:")
print(f"  Train styles: {sorted(train_df['style'].unique())}")
print(f"  Val styles: {sorted(val_df['style'].unique())}")
print(f"  Test styles: {sorted(test_df['style'].unique())}")

# Create caption dictionary
captions_dict = {}
for _, row in df_subset.iterrows():
    captions_dict[row['image_path']] = row['caption']

print(f"\nCaption dictionary created: {len(captions_dict)} entries")


Loading caption dataset...
Total images with captions: 13230

Style categories: 14
Styles: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']

Creating 50.0% stratified subset...
Subset size: 6615 (50.0%)

Creating train/val/test splits...
  Train: 70.0%, Val: 15.0%, Test: 15.0%

Split sizes:
  Train: 4630 (70.0%)
  Val: 992 (15.0%)
  Test: 993 (15.0%)

Style distribution check:
  Train styles: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']
  Val styles: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']
  Test styles: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']

Caption dictionary crea

## 3. Load Pre-trained Models


In [3]:
# Load CLIP model
import clip
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP model loaded!")

# Load FashionBERT
from transformers import AutoTokenizer, AutoModel
print("Loading FashionBERT...")
fashionbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
fashionbert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
fashionbert_model.eval()
print("FashionBERT model loaded!")


Loading CLIP model...
CLIP model loaded!
Loading FashionBERT...


2025-11-17 20:49:07.380676: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-17 20:49:07.402025: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FashionBERT model loaded!


2025-11-17 20:49:08.031275: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 4. Dataset and Model Classes


In [4]:
class FashionMultiModalDataset(Dataset):
    """Dataset class for multi-modal fashion style classification"""
    
    def __init__(self, df, captions_dict, style_to_idx, transform=None, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        self.transform = transform
        self.base_dir = base_dir
        
        # Filter out images without captions and store valid indices
        self.valid_indices = []
        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            image_path = row['image_path']
            # Convert to absolute path if needed
            if base_dir and not os.path.isabs(image_path):
                image_path = os.path.join(base_dir, image_path)
            # Decode URL-encoded characters
            if '%' in image_path:
                path_parts = image_path.split('/')
                decoded_parts = [unquote(part) if '%' in part else part for part in path_parts]
                image_path = '/'.join(decoded_parts)
            
            if image_path in captions_dict or row['image_path'] in captions_dict:
                self.valid_indices.append(idx)
        
        print(f"Dataset initialized with {len(self.valid_indices)} valid samples (out of {len(self.df)})")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        
        # Load image
        image_path = row['image_path']
        
        # Convert to absolute path if needed
        if self.base_dir and not os.path.isabs(image_path):
            image_path = os.path.join(self.base_dir, image_path)
        
        # Decode URL-encoded characters in filename
        if '%' in image_path:
            path_parts = image_path.split('/')
            decoded_parts = [unquote(part) if '%' in part else part for part in path_parts]
            image_path = '/'.join(decoded_parts)
        
        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert('RGB')
                if self.transform:
                    image = self.transform(image)
            else:
                # Return blank image if file doesn't exist
                image = torch.zeros(3, 224, 224)
        except Exception as e:
            # Return blank image if loading fails
            image = torch.zeros(3, 224, 224)
        
        # Get caption (try both paths)
        caption = self.captions_dict.get(image_path, 
                                        self.captions_dict.get(row['image_path'], 
                                                              "No caption available"))
        
        # Get label
        style = row['style']
        label = self.style_to_idx[style]
        
        return {
            'image': image,
            'caption': caption,
            'label': label,
            'style': style,
            'image_path': image_path
        }

class AttentionFusion(nn.Module):
    """
    Self-attention fusion mechanism (from archived notebook - proven to work better)
    Combines visual and textual features using self-attention
    """
    
    def __init__(self, visual_dim, textual_dim, hidden_dim=512):
        super(AttentionFusion, self).__init__()
        self.visual_dim = visual_dim
        self.textual_dim = textual_dim
        self.hidden_dim = hidden_dim
        
        # Project visual features
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)
        
        # Project textual features
        self.textual_proj = nn.Linear(textual_dim, hidden_dim)
        
        # Self-attention mechanism
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        
        # Layer normalization
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # Final projection
        self.final_proj = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, visual_features, textual_features):
        # Project features to common dimension
        visual_proj = self.visual_proj(visual_features)
        textual_proj = self.textual_proj(textual_features)
        
        # Stack features for attention
        combined_features = torch.stack([visual_proj, textual_proj], dim=1)  # [batch, 2, hidden_dim]
        
        # Apply self-attention
        attended_features, _ = self.attention(combined_features, combined_features, combined_features)
        
        # Layer normalization
        attended_features = self.layer_norm(attended_features)
        
        # Average pooling across modalities
        fused_features = torch.mean(attended_features, dim=1)  # [batch, hidden_dim]
        
        # Final projection
        fused_features = self.final_proj(fused_features)
        
        return fused_features

class MultiModalFashionClassifier(nn.Module):
    """Multi-modal fashion style classifier with configurable dropout"""
    
    def __init__(self, clip_model, fashionbert_model, num_classes, dropout=0.3, 
                 visual_dim=512, textual_dim=768):
        super(MultiModalFashionClassifier, self).__init__()
        
        self.clip_model = clip_model
        self.fashionbert_model = fashionbert_model
        self.visual_dim = visual_dim
        self.textual_dim = textual_dim
        
        # Fusion mechanism (self-attention)
        self.fusion = AttentionFusion(visual_dim, textual_dim)
        
        # Classification head with configurable dropout
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, images, captions):
        # Extract visual features using CLIP
        with torch.no_grad():
            visual_features = self.clip_model.encode_image(images)
            visual_features = visual_features.float()
        
        # Extract textual features using FashionBERT
        with torch.no_grad():
            inputs = fashionbert_tokenizer(
                captions, 
                return_tensors='pt', 
                padding=True, 
                truncation=True, 
                max_length=128
            ).to(device)
            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        # Fuse features
        fused_features = self.fusion(visual_features, textual_features)
        
        # Classification
        logits = self.classifier(fused_features)
        
        return logits

print("✅ Model classes defined!")


✅ Model classes defined!


## 5. Training Functions


In [5]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(train_loader, desc="Training", leave=False):
        images = batch['image'].to(device)
        captions = batch['caption']
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(images, captions)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), 100. * correct / total

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            images = batch['image'].to(device)
            captions = batch['caption']
            labels = batch['label'].to(device)
            
            logits = model(images, captions)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate macro F1
    val_macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0) if len(all_predictions) > 0 else 0.0
    
    return total_loss / len(val_loader), 100. * correct / total, all_predictions, all_labels, val_macro_f1

print("✅ Training functions defined!")


✅ Training functions defined!


## 6. Prepare Data Loaders and Class Weights


In [6]:
# Get base directory for absolute paths
BASE_DIR = os.path.abspath('.')

# Image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
print("Creating datasets...")
train_dataset = FashionMultiModalDataset(train_df, captions_dict, style_to_idx, transform, base_dir=BASE_DIR)
val_dataset = FashionMultiModalDataset(val_df, captions_dict, style_to_idx, transform, base_dir=BASE_DIR)
test_dataset = FashionMultiModalDataset(test_df, captions_dict, style_to_idx, transform, base_dir=BASE_DIR)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nData loaders created:")
print(f"  Train: {len(train_loader)} batches ({len(train_dataset)} samples)")
print(f"  Val: {len(val_loader)} batches ({len(val_dataset)} samples)")
print(f"  Test: {len(test_loader)} batches ({len(test_dataset)} samples)")

# Compute class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.array(list(style_to_idx.values())),
    y=train_df['style'].map(style_to_idx).values
)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f"\nClass weights computed for training")
print(f"  Min weight: {class_weights.min():.4f}, Max weight: {class_weights.max():.4f}")


Creating datasets...
Dataset initialized with 4630 valid samples (out of 4630)
Dataset initialized with 992 valid samples (out of 992)
Dataset initialized with 993 valid samples (out of 993)

Data loaders created:
  Train: 290 batches (4630 samples)
  Val: 62 batches (992 samples)
  Test: 63 batches (993 samples)

Class weights computed for training
  Min weight: 0.8546, Max weight: 1.1686


## 7. Experiment Runner Function


In [7]:
def run_experiment(config, train_loader, val_loader, test_loader, class_weights, 
                   clip_model, fashionbert_model, fashionbert_tokenizer, 
                   num_classes, device, all_styles):
    """
    Run a single experiment with given configuration
    
    Args:
        config: Dictionary with 'name', 'patience', 'dropout', 'weight_decay'
        train_loader, val_loader, test_loader: Data loaders
        class_weights: Class weights for loss
        clip_model, fashionbert_model: Pre-trained models
        fashionbert_tokenizer: BERT tokenizer
        num_classes: Number of classes
        device: Device
        all_styles: List of style names
    
    Returns:
        Dictionary with all results and metrics
    """
    
    print(f"\n{'='*60}")
    print(f"Running: {config['name']}")
    print(f"  Early Stopping Patience: {config['patience']}")
    print(f"  Dropout: {config['dropout']}")
    print(f"  Weight Decay: {config['weight_decay']}")
    print(f"{'='*60}")
    
    # Initialize model with config dropout
    model = MultiModalFashionClassifier(
        clip_model=clip_model,
        fashionbert_model=fashionbert_model,
        num_classes=num_classes,
        dropout=config['dropout']
    ).to(device)
    
    # Setup training
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LEARNING_RATE, 
        weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    
    # Training tracking
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    val_macro_f1s = []
    learning_rates = []
    
    best_val_macro_f1 = 0
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    early_stopped = False
    
    start_time = time.time()
    
    # Training loop with early stopping
    for epoch in range(NUM_EPOCHS):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, val_preds, val_labels, val_macro_f1 = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update scheduler
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])
        
        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        val_macro_f1s.append(val_macro_f1)
        
        # Track best Macro F1 (for saving & early stopping)
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), 
                      os.path.join(ARTIFACTS_DIR, "models", f"{config['name']}_best_model.pth"))
        else:
            patience_counter += 1
        
        # Track best loss (for overfitting detection)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
        
        # Early stopping (based on Macro F1)
        if patience_counter >= config['patience']:
            early_stopped = True
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {config['patience']} epochs)")
            break
        
        # Print progress
        if (epoch + 1) % 3 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS}: "
                  f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
                  f"Val Macro F1={val_macro_f1:.4f}")
    
    total_time = time.time() - start_time
    
    # Load best model for final evaluation
    model.load_state_dict(torch.load(
        os.path.join(ARTIFACTS_DIR, "models", f"{config['name']}_best_model.pth")))
    model.eval()
    
    # Final evaluation on test set
    test_loss, test_acc, test_preds, test_labels, test_macro_f1 = validate_epoch(
        model, test_loader, criterion, device
    )
    
    # Detect overfitting
    # Overfitting if validation loss increases by more than 5% after best epoch
    if len(val_losses) > best_epoch:
        val_loss_after_best = min(val_losses[best_epoch:])  # Best loss after best epoch
        overfitting_detected = val_loss_after_best > best_val_loss * 1.05
    else:
        overfitting_detected = False
    
    # Calculate train/val gap at best epoch
    train_val_gap = train_losses[best_epoch - 1] - best_val_loss if best_epoch > 0 else 0
    
    # Create learning curves plot
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss curves
    axes[0, 0].plot(train_losses, label='Train Loss', color='blue', linewidth=2)
    axes[0, 0].plot(val_losses, label='Val Loss', color='red', linewidth=2)
    axes[0, 0].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch {best_epoch}')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[0, 1].plot(train_accs, label='Train Acc', color='blue', linewidth=2)
    axes[0, 1].plot(val_accs, label='Val Acc', color='red', linewidth=2)
    axes[0, 1].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7)
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Macro F1 curve
    axes[0, 2].plot(val_macro_f1s, label='Val Macro F1', color='green', linewidth=2)
    axes[0, 2].axvline(x=best_epoch-1, color='red', linestyle='--', alpha=0.7)
    axes[0, 2].axhline(y=best_val_macro_f1, color='red', linestyle='--', alpha=0.7, 
                       label=f'Best: {best_val_macro_f1:.4f}')
    axes[0, 2].set_title('Validation Macro F1-Score')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Macro F1')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Learning rate
    axes[1, 0].plot(learning_rates, label='Learning Rate', color='purple', linewidth=2)
    axes[1, 0].set_title('Learning Rate Schedule')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Loss difference (overfitting indicator)
    loss_diff = [t - v for t, v in zip(train_losses, val_losses)]
    axes[1, 1].plot(loss_diff, label='Train - Val Loss', color='orange', linewidth=2)
    axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 1].set_title('Train-Val Loss Gap (Overfitting Indicator)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss Difference')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Summary text
    axes[1, 2].axis('off')
    summary_text = f"""
Configuration: {config['name']}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Early Stopping: {config['patience']}
Dropout: {config['dropout']}
Weight Decay: {config['weight_decay']}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Epoch: {best_epoch}
Early Stopped: {early_stopped}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Val Loss: {best_val_loss:.4f}
Final Val Loss: {val_losses[-1]:.4f}
Overfitting: {'Yes' if overfitting_detected else 'No'}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Val Macro F1: {best_val_macro_f1:.4f}
Test Macro F1: {test_macro_f1:.4f}
Test Accuracy: {test_acc:.2f}%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Training Time: {total_time/60:.1f} minutes
    """
    axes[1, 2].text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
                    verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle(f'Learning Curves: {config["name"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(ARTIFACTS_DIR, "experiments", f"{config['name']}_learning_curves.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Prepare results dictionary
    results = {
        "experiment_id": config['name'],
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "patience": config['patience'],
            "dropout": config['dropout'],
            "weight_decay": float(config['weight_decay']),
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "max_epochs": NUM_EPOCHS,
            "random_seed": RANDOM_SEED
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "best_epoch": best_epoch,
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60)
        },
        "validation_metrics": {
            "best_val_macro_f1": float(best_val_macro_f1),
            "best_val_accuracy": float(val_accs[best_epoch - 1]) if best_epoch > 0 else 0.0,
            "best_val_loss": float(best_val_loss),
            "final_val_macro_f1": float(val_macro_f1s[-1]),
            "final_val_accuracy": float(val_accs[-1]),
            "final_val_loss": float(val_losses[-1])
        },
        "test_metrics": {
            "test_macro_f1": float(test_macro_f1),
            "test_accuracy": float(test_acc),
            "test_loss": float(test_loss)
        },
        "overfitting_analysis": {
            "overfitting_detected": overfitting_detected,
            "best_val_loss": float(best_val_loss),
            "val_loss_after_best": float(val_losses[best_epoch:][0]) if len(val_losses) > best_epoch else float(val_losses[-1]),
            "increase_percentage": float((val_losses[best_epoch:][0] - best_val_loss) / best_val_loss * 100) if len(val_losses) > best_epoch else 0.0,
            "train_val_gap": float(train_val_gap)
        },
        "training_curves": {
            "train_losses": [float(x) for x in train_losses],
            "val_losses": [float(x) for x in val_losses],
            "train_accs": [float(x) for x in train_accs],
            "val_accs": [float(x) for x in val_accs],
            "val_macro_f1s": [float(x) for x in val_macro_f1s],
            "learning_rates": [float(x) for x in learning_rates]
        }
    }
    
    # Save results JSON
    json_path = os.path.join(METRICS_DIR, f"{config['name']}_results.json")
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"  ✅ Completed: Best Val Macro F1={best_val_macro_f1:.4f}, "
          f"Overfitting={'Yes' if overfitting_detected else 'No'}")
    
    return results

print("✅ Experiment runner function defined!")


✅ Experiment runner function defined!


In [8]:
# Run all 8 experiments
all_results = []

print(f"\n{'='*70}")
print(f"STARTING EXPERIMENT SERIES")
print(f"Total experiments: {len(EXPERIMENTS)}")
print(f"Estimated time: ~{len(EXPERIMENTS) * 1.25:.1f} hours (on 50% subset)")
print(f"{'='*70}")

for i, config in enumerate(EXPERIMENTS):
    print(f"\n{'#'*70}")
    print(f"Experiment {i+1}/{len(EXPERIMENTS)}")
    print(f"{'#'*70}")
    
    result = run_experiment(
        config=config,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        class_weights=class_weights,
        clip_model=clip_model,
        fashionbert_model=fashionbert_model,
        fashionbert_tokenizer=fashionbert_tokenizer,
        num_classes=num_classes,
        device=device,
        all_styles=all_styles
    )
    
    all_results.append(result)
    
    # Progress update
    remaining = len(EXPERIMENTS) - (i + 1)
    print(f"\n✅ Progress: {i+1}/{len(EXPERIMENTS)} completed, {remaining} remaining")

print(f"\n{'='*70}")
print(f"ALL EXPERIMENTS COMPLETED!")
print(f"{'='*70}")



STARTING EXPERIMENT SERIES
Total experiments: 8
Estimated time: ~10.0 hours (on 50% subset)

######################################################################
Experiment 1/8
######################################################################

Running: ES5_Drop0.4_WD1e-5
  Early Stopping Patience: 5
  Dropout: 0.4
  Weight Decay: 1e-05


  Epoch 1/20: Train Loss=1.8021, Val Loss=0.8511, Val Macro F1=0.7282


  Epoch 3/20: Train Loss=0.7125, Val Loss=0.5674, Val Macro F1=0.8005


  Epoch 6/20: Train Loss=0.4844, Val Loss=0.5601, Val Macro F1=0.8063


  Epoch 9/20: Train Loss=0.3888, Val Loss=0.5596, Val Macro F1=0.8157


...

  Epoch 9/20: Train Loss=0.4526, Val Loss=0.5680, Val Macro F1=0.8089


  Epoch 12/20: Train Loss=0.3909, Val Loss=0.5825, Val Macro F1=0.8077


  Epoch 15/20: Train Loss=0.3079, Val Loss=0.6390, Val Macro F1=0.8089


  Epoch 18/20: Train Loss=0.2820, Val Loss=0.6529, Val Macro F1=0.8010


  ✅ Completed: Best Val Macro F1=0.8128, Overfitting=Yes

✅ Progress: 8/8 completed, 0 remaining

ALL EXPERIMENTS COMPLETED!


## 9. Generate Summary Table and Comparison


In [ ]:
# Create summary DataFrame
summary_data = []
for result in all_results:
    summary_data.append({
        "Config": result['experiment_id'],
        "Patience": result['configuration']['patience'],
        "Dropout": result['configuration']['dropout'],
        "Weight Decay": result['configuration']['weight_decay'],
        "Best Epoch": result['training_info']['best_epoch'],
        "Early Stopped": result['training_info']['early_stopped'],
        "Best Val Loss": f"{result['validation_metrics']['best_val_loss']:.4f}",
        "Final Val Loss": f"{result['validation_metrics']['final_val_loss']:.4f}",
        "Overfitting": "Yes" if result['overfitting_analysis']['overfitting_detected'] else "No",
        "Best Macro F1": f"{result['validation_metrics']['best_val_macro_f1']:.4f}",
        "Test Macro F1": f"{result['test_metrics']['test_macro_f1']:.4f}",
        "Test Accuracy": f"{result['test_metrics']['test_accuracy']:.2f}%",
        "Train/Val Gap": f"{result['overfitting_analysis']['train_val_gap']:.4f}",
        "Time (min)": f"{result['training_info']['total_time_minutes']:.1f}"
    })

df_summary = pd.DataFrame(summary_data)

# Sort by: Train/Val Gap (smaller absolute value = better), then best Macro F1
df_summary_sorted = df_summary.copy()
df_summary_sorted['Train/Val Gap_Num'] = df_summary_sorted['Train/Val Gap'].astype(float).abs()
df_summary_sorted['Best Macro F1_Num'] = df_summary_sorted['Best Macro F1'].astype(float)
df_summary_sorted = df_summary_sorted.sort_values(
    by=['Train/Val Gap_Num', 'Best Macro F1_Num'],
    ascending=[True, False]  # Smaller gap first, then higher F1
).drop(columns=['Train/Val Gap_Num', 'Best Macro F1_Num'])

print("\n" + "="*80)
print("SUMMARY TABLE - All Configurations")
print("="*80)
print(df_summary.to_string(index=False))

print("\n" + "="*80)
print("SORTED BY BEST CONFIGURATION (Smallest Train/Val Gap → Best Macro F1)")
print("="*80)
print(df_summary_sorted.to_string(index=False))

# Save to CSV
csv_path = os.path.join(METRICS_DIR, "summary_table.csv")
df_summary_sorted.to_csv(csv_path, index=False)
print(f"\n✅ Summary table saved to: {csv_path}")

# Identify top configuration
top_config = df_summary_sorted.iloc[0]
print(f"\n{'='*80}")
print("✅ RECOMMENDED CONFIGURATION (Smallest Train/Val Gap)")
print("="*80)
print(f"   Config: {top_config['Config']}")
print(f"   Train/Val Gap: {top_config['Train/Val Gap']}")
print(f"   Best Macro F1: {top_config['Best Macro F1']}")
print(f"   Test Macro F1: {top_config['Test Macro F1']}")
print(f"   Test Accuracy: {top_config['Test Accuracy']}")
print(f"   Best Epoch: {top_config['Best Epoch']}")
print(f"   Overfitting Flag: {top_config['Overfitting']}")
print(f"{'='*80}")



SUMMARY TABLE - All Configurations
            Config  Patience  Dropout  Weight Decay  Best Epoch  Early Stopped Best Val Loss Final Val Loss Overfitting Best Macro F1 Test Macro F1 Test Accuracy Train/Val Gap Time (min)
ES5_Drop0.4_WD1e-5         5      0.4       0.00001           9           True        0.5596         0.6373          No        0.8157        0.8071        80.77%       -0.1708        1.8
ES5_Drop0.4_WD1e-4         5      0.4       0.00010           9           True        0.5385         0.6677         Yes        0.8119        0.8049        80.56%       -0.1690        1.8
ES5_Drop0.5_WD1e-5         5      0.5       0.00001           8           True        0.5441         0.6232         Yes        0.8134        0.8011        80.26%       -0.0314        1.7
ES5_Drop0.5_WD1e-4         5      0.5       0.00010           9           True        0.5563         0.6554         Yes        0.8186        0.8140        81.27%       -0.0951        1.8
ES7_Drop0.4_WD1e-5         7 

## 10. Comparison Visualization


In [10]:
# Create comparison plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: All Val Loss curves
for result in all_results:
    overfitting = result['overfitting_analysis']['overfitting_detected']
    color = 'red' if overfitting else 'green'
    alpha = 0.6 if overfitting else 0.8
    axes[0, 0].plot(result['training_curves']['val_losses'], 
                    label=result['experiment_id'], color=color, alpha=alpha, linewidth=2)
axes[0, 0].set_title('Validation Loss Comparison - All Configurations', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Validation Loss')
axes[0, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: All Macro F1 scores
for result in all_results:
    overfitting = result['overfitting_analysis']['overfitting_detected']
    color = 'red' if overfitting else 'green'
    alpha = 0.6 if overfitting else 0.8
    axes[0, 1].plot(result['training_curves']['val_macro_f1s'], 
                    label=result['experiment_id'], color=color, alpha=alpha, linewidth=2)
axes[0, 1].set_title('Validation Macro F1 Comparison - All Configurations', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Macro F1-Score')
axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Best Macro F1 by configuration
best_f1s = [result['validation_metrics']['best_val_macro_f1'] for result in all_results]
config_names = [result['experiment_id'] for result in all_results]
overfitting_flags = [result['overfitting_analysis']['overfitting_detected'] for result in all_results]
colors = ['red' if of else 'green' for of in overfitting_flags]

bars = axes[1, 0].bar(range(len(config_names)), best_f1s, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].set_xticks(range(len(config_names)))
axes[1, 0].set_xticklabels(config_names, rotation=45, ha='right', fontsize=8)
axes[1, 0].set_ylabel('Best Validation Macro F1')
axes[1, 0].set_title('Best Macro F1 by Configuration\n(Green=No Overfitting, Red=Overfitting)', 
                     fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, best_f1s)):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# Plot 4: Train-Val Gap
train_val_gaps = [result['overfitting_analysis']['train_val_gap'] for result in all_results]
bars = axes[1, 1].bar(range(len(config_names)), train_val_gaps, color=colors, alpha=0.7, edgecolor='black')
axes[1, 1].set_xticks(range(len(config_names)))
axes[1, 1].set_xticklabels(config_names, rotation=45, ha='right', fontsize=8)
axes[1, 1].set_ylabel('Train-Val Loss Gap')
axes[1, 1].set_title('Train-Val Loss Gap by Configuration\n(Green=No Overfitting, Red=Overfitting)', 
                     fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, train_val_gaps)):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Overfitting Fix Testing - Comparison of All Configurations', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()

# Save comparison plot
comparison_path = os.path.join(ARTIFACTS_DIR, "comparison_plot.png")
plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"✅ Comparison plot saved to: {comparison_path}")
print("\n✅ All results generated successfully!")


✅ Comparison plot saved to: overfitting_test_results_50pct/comparison_plot.png

✅ All results generated successfully!
